## Web Scraping Capstone
### ReadBridge Nigeria — Book Catalogue Pipeline
#### 🎯  Pagination · PostgreSQL · GitHub




##### ReadBridge Nigeria is a Lagos-based EdTech startup on a mission to help Nigerian university students discover and access affordable international books and academic resources.

##### They are building Nigeria's first curated digital book catalogue — a searchable database covering thousands of titles, prices, genres, and availability statuses.

##### Problem: Before they can afford licensed publisher APIs, they need to bootstrap their catalogue using publicly available data.


##### Import everything needed

In [1]:
import pandas as pd
import requests
import os
import time
from bs4 import BeautifulSoup
import sqlite3
from dotenv import load_dotenv
from sqlalchemy import create_engine
from urllib.parse import quote_plus

print("All libraries imported successfully.")

All libraries imported successfully.


In [2]:
# fetch pages with requests and BeautifulSoup

url = 'https://books.toscrape.com'
response = requests.get(url)

print(f"Status code: {response.status_code}")
print(f"Content type: {response.headers['Content-Type']}")
print(f"Content Headers: {response.headers}")
print(f'Page size   : {len(response.text):,} characters')

Status code: 200
Content type: text/html
Content Headers: {'Date': 'Tue, 06 Oct 2026 07:38:23 GMT', 'Content-Type': 'text/html', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Last-Modified': 'Wed, 08 Feb 2023 21:02:32 GMT', 'ETag': 'W/"63e40de8-c85e"', 'Strict-Transport-Security': 'max-age=0; includeSubDomains; preload', 'Content-Encoding': 'br'}
Page size   : 51,294 characters


In [3]:
# ── Always check status before scraping ──────────────────────────────────
def safe_get(url, delay=1):
    """Fetch a URL, return None if it fails, and sleep politely after."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()    # Raises exception for 4xx / 5xx codes
        time.sleep(delay)              # Polite pause — never hammer a server
        return response
    except requests.RequestException as e:
        print(f'Request failed: {e}')
        return None

resp = safe_get('https://books.toscrape.com')
if resp:
    print(f'Got {len(resp.text):,} chars')

Got 51,294 chars


### Real Work Now

### We Begin with Pagination

#### Test the URL

In [4]:

url = 'https://books.toscrape.com/catalogue/page-{}.html'

# Test it manually first
test_url = url.format(1)
print('Page 1 URL:', test_url)
print('Page 2 URL:', url.format(2))
print('Page 3 URL:', url.format(3))
print('Page 4 URL:', url.format(4))
print('Page 5 URL:', url.format(5))

Page 1 URL: https://books.toscrape.com/catalogue/page-1.html
Page 2 URL: https://books.toscrape.com/catalogue/page-2.html
Page 3 URL: https://books.toscrape.com/catalogue/page-3.html
Page 4 URL: https://books.toscrape.com/catalogue/page-4.html
Page 5 URL: https://books.toscrape.com/catalogue/page-5.html


#### Now we Inspect

In [5]:

resp = safe_get('https://books.toscrape.com/catalogue/page-1.html')
soup = BeautifulSoup(resp.text, 'html.parser')

# Each book is in <article class='product_pod'>
books_on_page = soup.find_all('article', class_='product_pod')
print(f'Books on page 1: {len(books_on_page)}')

# Preview first book structure
print('\n--- First book HTML ---')
print(books_on_page[0].prettify()[:500])

Books on page 1: 20

--- First book HTML ---
<article class="product_pod">
 <div class="image_container">
  <a href="a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="../media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="a-light-in-the-attic_1000/index.html" title="A


In [6]:
# ── Extract from one book
book = books_on_page[0]

title  = book.find('h3').find('a')['title']        # title attribute on the <a> tag
price  = book.find('p', class_='price_color').text.strip()
rating = book.find('p', class_='star-rating')['class'][1]   # 'One', 'Two', 'Three'...
avail  = book.find('p', class_='instock').get_text(strip=True)

print(f'Title  : {title}')
print(f'Price  : {price}')
print(f'Rating : {rating}')
print(f'Stock  : {avail}')

Title  : A Light in the Attic
Price  : Â£51.77
Rating : Three
Stock  : In stock


In [7]:
# Full pagination loop — 10 pages
# for loop to scrape all 10 pages of books

url  = 'https://books.toscrape.com/catalogue/page-{}.html'
all_books = []

for page_num in range(1, 11):      # Pages 1 to 10
    print(f'Fetching page {page_num}...')

    resp = safe_get(url.format(page_num), delay=1)
    if resp is None:
        print(f'Skipping page {page_num} — request failed')
        continue

    soup = BeautifulSoup(resp.text, 'html.parser')

    for book in soup.find_all('article', class_='product_pod'): # find all books on the page
        title  = book.find('h3').find('a')['title'] # find the title of the book
        price  = book.find('p', class_='price_color').text.strip()  # find the price of the book
        rating = book.find('p', class_='star-rating')['class'][1]   # find the rating of the book
        avail  = book.find('p', class_='instock').get_text(strip=True)  # find the availability of the book

        all_books.append({  # append the book data to the list
            'title'       : title,
            'price'       : price,
            'rating'      : rating,
            'availability': avail,
            'page'        : page_num
        })



Fetching page 1...
Fetching page 2...
Fetching page 3...
Request failed: HTTPSConnectionPool(host='books.toscrape.com', port=443): Max retries exceeded with url: /catalogue/page-3.html (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001E549882490>, 'Connection to books.toscrape.com timed out. (connect timeout=10)'))
Skipping page 3 — request failed
Fetching page 4...
Fetching page 5...
Fetching page 6...
Fetching page 7...
Fetching page 8...
Fetching page 9...
Fetching page 10...


In [8]:
# Load to DataFrame and preview
books_df = pd.DataFrame(all_books)
print(f'\nScraped {len(books_df)} books across 10 pages')
books_df.head(10)



Scraped 180 books across 10 pages


,title,price,rating,availability,page
0,A Light in the Attic,Â£51.77,Three,In stock,1
1,Tipping the Velvet,Â£53.74,One,In stock,1
2,Soumission,Â£50.10,One,In stock,1
3,Sharp Objects,Â£47.82,Four,In stock,1
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,1
5,The Requiem Red,Â£22.65,One,In stock,1
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,Four,In stock,1
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,Three,In stock,1
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,Four,In stock,1
9,The Black Maria,Â£52.15,One,In stock,1


In [9]:
# Save and inspect 
books_df.to_csv('books_10pages.csv', index=False)
print('Saved to books_10pages.csv')

Saved to books_10pages.csv


In [10]:
# create a copy of the DataFrame to avoid modifying the original
books_df_copy = books_df.copy()

### Clean and Transform

In [11]:
# Distribution by rating
print('\nBooks by rating:')
print(books_df_copy['rating'].value_counts())


Books by rating:
rating
One      43
Two      39
Four     34
Five     33
Three    31
Name: count, dtype: int64


In [12]:
# Price column is a string. We need to clean it and convert it to a float type for analysis.
books_df_copy['price'] = books_df_copy['price'].str.replace('Â£', '')


In [13]:
books_df_copy

,title,price,rating,availability,page
0,A Light in the Attic,51.77,Three,In stock,1
1,Tipping the Velvet,53.74,One,In stock,1
2,Soumission,50.10,One,In stock,1
3,Sharp Objects,47.82,Four,In stock,1
4,Sapiens: A Brief History of Humankind,54.23,Five,In stock,1
...,...,...,...,...,...
175,Eureka Trivia 6.0,54.59,Four,In stock,10
176,Drive: The Surprising Truth About What Motivat...,34.95,Four,In stock,10
177,Done Rubbed Out (Reightman & Bailey #1),37.72,Five,In stock,10
178,Doing It Over (Most Likely To #1),35.61,Three,In stock,10


In [14]:
books_df_copy['price'].dtype

dtype('O')

#### Convert rating to numeric

In [15]:
rating_dict = {
    'One': 1,
    'Two': 2, 
    'Three': 3, 
    'Four': 4, 
    'Five': 5
    }

In [16]:
books_df_copy['rating'] = (books_df_copy['rating'].replace(rating_dict).astype("Int64"))


C:\Users\hp\AppData\Local\Temp\ipykernel_37456\4193359606.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  books_df_copy['rating'] = (books_df_copy['rating'].replace(rating_dict).astype("Int64"))


In [17]:
# Add scraped at columns to track when the data was collected
books_df_copy['scraped_at'] = pd.Timestamp.now()

In [18]:
books_df_copy["scraped_at"] = pd.Timestamp.now().replace(second=0, microsecond=0)

##### convert all columns to the right datatypes

In [19]:
books_df_copy = books_df_copy.astype({
    'title' : 'str',
    'price' : 'float',
    'rating' : 'Int64',
    'availability' : 'str',
    'page' : 'Int64',
    'scraped_at' : 'datetime64[ns]' 
})


In [20]:
books_df_copy['scraped_at'].head(5)

0   2026-10-06 07:39:00
1   2026-10-06 07:39:00
2   2026-10-06 07:39:00
3   2026-10-06 07:39:00
4   2026-10-06 07:39:00
Name: scraped_at, dtype: datetime64[ns]

In [21]:
books_df_clean = books_df_copy
books_df_clean

,title,price,rating,availability,page,scraped_at
0,A Light in the Attic,51.77,3,In stock,1,2026-10-06 07:39:00
1,Tipping the Velvet,53.74,1,In stock,1,2026-10-06 07:39:00
2,Soumission,50.10,1,In stock,1,2026-10-06 07:39:00
3,Sharp Objects,47.82,4,In stock,1,2026-10-06 07:39:00
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,1,2026-10-06 07:39:00
...,...,...,...,...,...,...
175,Eureka Trivia 6.0,54.59,4,In stock,10,2026-10-06 07:39:00
176,Drive: The Surprising Truth About What Motivat...,34.95,4,In stock,10,2026-10-06 07:39:00
177,Done Rubbed Out (Reightman & Bailey #1),37.72,5,In stock,10,2026-10-06 07:39:00
178,Doing It Over (Most Likely To #1),35.61,3,In stock,10,2026-10-06 07:39:00


#### Load to PostgreSQL

In [22]:
load_dotenv()  # Load environment variables from .env file

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql://{user}:{quote_plus(password)}@{host}:{port}/{db}"
)

## Write the DataFrame to the database
books_df_clean.to_sql(name='books_catalogue', con=engine, if_exists='replace', index=False)

print("DataFrame written to database successfully.")

DataFrame written to database successfully.


In [23]:
books_db = pd.read_sql('SELECT * FROM books_catalogue', con=engine)

books_db.head(5)

,title,price,rating,availability,page,scraped_at
0,A Light in the Attic,51.77,3,In stock,1,2026-10-06 07:39:00
1,Tipping the Velvet,53.74,1,In stock,1,2026-10-06 07:39:00
2,Soumission,50.10,1,In stock,1,2026-10-06 07:39:00
3,Sharp Objects,47.82,4,In stock,1,2026-10-06 07:39:00
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,1,2026-10-06 07:39:00
